# 02 · Exploratory analysis and return behaviour

**Questions:** How has IREN performed against Bitcoin, the Nasdaq and its peers? How risky is it? Do its returns behave the way standard models assume (normal, independent, stationary)?

> **Finance note: why returns, not prices?** Prices trend and are non-stationary, so regressions on prices produce *spurious* results. Returns are roughly stationary. Log returns also add up over time: the sum of daily log returns is the period's log return.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import adfuller, kpss
from iren import config, metrics, plotting

plotting.set_style()
C = plotting.ASSET_COLORS
pd.set_option("display.float_format", "{:,.3f}".format)

In [ ]:
prices = pd.read_csv(config.DATA_PROCESSED / "prices.csv", index_col=0, parse_dates=True)
rf = pd.read_csv(config.DATA_PROCESSED / "risk_free.csv", index_col=0, parse_dates=True)["rf_daily"]

rets = metrics.log_returns(prices)
# Equal-weight peer baskets (average of whichever members were listed that day)
rets["AI peers"] = rets[config.PEERS_AI_CLOUD].mean(axis=1, skipna=True)
rets["Miners"] = rets[config.PEERS_PURE_MINERS].mean(axis=1, skipna=True)
FOCUS = ["IREN", "BTC-USD", "QQQ", "AI peers", "Miners"]
rets = rets.loc[rets["IREN"].first_valid_index():]
levels = np.exp(rets[FOCUS].fillna(0).cumsum())  # growth of $1
rets[FOCUS].describe().T

## 1. Performance and risk summary

- **Sharpe** = excess return ÷ volatility (annualised). **Sortino** only counts downside volatility, which is fairer for assets with big upside jumps.
- **Max drawdown** is the worst peak-to-trough fall an investor who bought at the top would have lived through.

In [ ]:
summary = metrics.summary_stats(rets[FOCUS], rf_daily=rf, prices=levels)
summary.style.format({"obs": "{:.0f}", "ann_return": "{:.1%}", "ann_vol": "{:.1%}",
                      "max_drawdown": "{:.1%}", "sharpe": "{:.2f}", "sortino": "{:.2f}",
                      "skew": "{:.2f}", "excess_kurtosis": "{:.1f}"})

In [ ]:
fig, ax = plt.subplots()
for col in FOCUS:
    ax.plot(levels.index, levels[col], label=col, color=C[col], lw=2.2 if col == "IREN" else 1.4)
ax.set_yscale("log")
ax.axvline(pd.Timestamp(config.PIVOT_DATE), color=plotting.INK_2, ls="--", lw=1)
ax.annotate("Microsoft contract", (pd.Timestamp(config.PIVOT_DATE), ax.get_ylim()[0]),
            xytext=(-4, 6), textcoords="offset points", ha="right", fontsize=9, color=plotting.INK_2)
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.FuncFormatter(lambda v, _: f"${v:,.2g}"))
ax.yaxis.set_major_locator(plt.matplotlib.ticker.LogLocator(base=10, subs=(1, 2, 5)))
ax.yaxis.set_minor_formatter(plt.matplotlib.ticker.NullFormatter())
ax.set_title("Growth of $1 since IREN's listing (log scale)")
ax.set_ylabel("Value of $1")
ax.legend(ncol=5, loc="upper left")
plotting.save(fig, "02_growth_of_1")

In [ ]:
dd = metrics.drawdown_series(prices["IREN"])
mdd, peak, trough = metrics.max_drawdown(prices["IREN"])
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.fill_between(dd.index, dd, 0, color=C["IREN"], alpha=0.25, lw=0)
ax.plot(dd.index, dd, color=C["IREN"], lw=1)
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
ax.set_title(f"IREN drawdown from previous peak (worst: {mdd:.0%}, {peak:%b %Y} to {trough:%b %Y})")
plotting.save(fig, "02_iren_drawdown")

In [ ]:
rv = metrics.rolling_vol(rets[["IREN", "BTC-USD", "QQQ"]], window=63)
fig, ax = plt.subplots()
for col in rv:
    ax.plot(rv.index, rv[col], label=col, color=C[col])
ax.axvline(pd.Timestamp(config.PIVOT_DATE), color=plotting.INK_2, ls="--", lw=1)
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
ax.set_title("Rolling 63-day annualised volatility")
ax.legend(loc="upper left")
plotting.save(fig, "02_rolling_vol")

## 2. Stylised facts: are returns normal, independent, stationary?

| Test | H0 | If rejected, it means… | Used later in |
|---|---|---|---|
| Jarque–Bera | returns are normal | fat tails → normal VaR is too low | 05 (Student-t GARCH, Cornish–Fisher) |
| Ljung–Box on r | no autocorrelation | there is a predictable mean component | 05 (ARIMA mean) |
| Ljung–Box on r² | no autocorrelation in squared returns | volatility clusters | 05 (GARCH) |
| ADF | unit root (non-stationary) | series is stationary | 04 (regress returns, not prices) |
| KPSS | stationary | series is non-stationary | the opposite H0 to ADF, so use both |

In [ ]:
r = rets["IREN"].dropna()
logp = np.log(prices["IREN"].dropna())
jb = stats.jarque_bera(r)
lb_r = acorr_ljungbox(r, lags=[10], return_df=True).iloc[0]
lb_r2 = acorr_ljungbox(r**2, lags=[10], return_df=True).iloc[0]
adf_p, adf_r = adfuller(logp, autolag="AIC")[1], adfuller(r, autolag="AIC")[1]
kpss_p, kpss_r = kpss(logp, nlags="auto")[1], kpss(r, nlags="auto")[1]

tests = pd.DataFrame([
    ["Jarque-Bera", "returns", jb.statistic, jb.pvalue],
    ["Ljung-Box (10)", "returns", lb_r.lb_stat, lb_r.lb_pvalue],
    ["Ljung-Box (10)", "squared returns", lb_r2.lb_stat, lb_r2.lb_pvalue],
    ["ADF", "log price", np.nan, adf_p], ["ADF", "returns", np.nan, adf_r],
    ["KPSS", "log price", np.nan, kpss_p], ["KPSS", "returns", np.nan, kpss_r],
], columns=["test", "series", "statistic", "p_value"])
tests["reject_H0_at_5%"] = tests["p_value"] < 0.05
tests  # note: KPSS p-values are capped at 0.01 / 0.10 by statsmodels (a warning is normal)

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4))
a1.hist(r, bins=80, density=True, color=C["IREN"], alpha=0.6, label="IREN daily log returns")
x = np.linspace(r.min(), r.max(), 400)
a1.plot(x, stats.norm.pdf(x, r.mean(), r.std()), color=plotting.INK, lw=1.2, label="Normal, same mean/sd")
a1.set_title("Fat tails vs the normal curve"); a1.legend(fontsize=8)
stats.probplot((r - r.mean()) / r.std(), dist="norm", plot=a2)
a2.get_lines()[0].set(color=C["IREN"], markersize=3); a2.get_lines()[1].set(color=plotting.INK)
a2.set_title("Q-Q plot: points off the line = fat tails")
a2.set_xlabel("Normal quantiles"); a2.set_ylabel("IREN standardised returns")
a1.set_xlabel("Daily log return"); a1.set_ylabel("Density")
plotting.save(fig, "02_return_distribution")

## 3. Preview of H1 and H2: did IREN's behaviour change after the pivot?

Compare correlations before and after the Microsoft contract. A **Fisher z-test** checks whether a change in correlation is statistically significant. Notebook 04 does this properly with regressions and a Chow test.

In [ ]:
def fisher_z_test(r1, n1, r2, n2):
    """Two-sided test of H0: rho_1 = rho_2 for independent samples."""
    z = (np.arctanh(r1) - np.arctanh(r2)) / np.sqrt(1 / (n1 - 3) + 1 / (n2 - 3))
    return z, 2 * (1 - stats.norm.cdf(abs(z)))

pivot = pd.Timestamp(config.PIVOT_DATE)
pre, post = rets.loc[:pivot - pd.Timedelta(days=1)], rets.loc[pivot:]
rows = []
for other in ["BTC-USD", "QQQ", "AI peers", "Miners"]:
    a, b = pre[["IREN", other]].dropna(), post[["IREN", other]].dropna()
    c1, c2 = a.corr().iloc[0, 1], b.corr().iloc[0, 1]
    z, p = fisher_z_test(c1, len(a), c2, len(b))
    rows.append([other, c1, len(a), c2, len(b), z, p])
corr_shift = pd.DataFrame(rows, columns=["vs", "corr_pre", "n_pre", "corr_post", "n_post", "z", "p_value"])
corr_shift["vol_pre"] = pre["IREN"].std() * np.sqrt(252)
corr_shift["vol_post"] = post["IREN"].std() * np.sqrt(252)
corr_shift

> **Caveat to write in your report:** the AI-peer basket has few members before 2025 (CoreWeave and Nebius listed late), so the pre-pivot correlation with *AI peers* rests on a thinner basket. Mention it; don't hide it.

## So what for the investor?

*Write 3-5 sentences in plain English. Use numbers from this notebook. Notebook 09 is built from these cells, so make each sentence count.*

- 
- 
- 

## Commit checkpoint

1. **Kernel → Restart → Run All** and check there are no errors.
2. Source Control panel (Ctrl+Shift+G): review the diff, then commit with messages like:
   - `feat: compute returns, drawdowns and Sharpe ratios`
   - `feat: add normality, Ljung-Box, ADF and KPSS tests`
   - `feat: preview correlation shift around the pivot`
3. `git push -u origin nb02-eda`, then open a pull request on GitHub and merge it.
